# Initialization

Case study parameters are available on [GitHub](https://github.com/thomaslu678/Praxis-Lab-25-26/tree/main/assets). This information determines the extent of the needed meteorological station data.

## Expected Feature Columns

* name (str)
* start (int)
* end (int)
* similarity (float)
* station (str)

In [ ]:
import requests
import pandas as pd
from google.colab import files
from io import StringIO

project_name = "gee-481701"
studies_path = "https://raw.githubusercontent.com/thomaslu678/Praxis-Lab-25-26/refs/heads/main/assets/studies/studies.csv"

r = requests.get(studies_path)

r.raise_for_status()

studies = pd.read_csv(StringIO(r.text))

HTTPError: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/thomaslu678/Praxis-Lab-25-26/refs/heads/main/assets/studies.csv

# Meteorological Station Data Fetching

Using data made available by the National Oceanic and Atmospheric Administration, hourly dry-bulb temperature and relative humidity data are acquired between 5 years before and 5 years after a case study, for the months of June, July, and August.

In [ ]:
# kwargs used to make function tolerant of extra arguments
def get_data(start: int, end: int, station: str, **kwargs) -> pd.DataFrame:
  print(station)
  r = requests.get(
      "https://www.ncei.noaa.gov/access/services/data/v1",
      params = {
          "dataset": "global-historical-climatology-network-hourly",
          "format": "csv", # json was being troublesome; using csv instead
          "units": "metric",
          "dataTypes": "STATION,Year,Month,Day,Hour,temperature,relative_humidity",
          "stations": station,
          "startDate": f"{start - 5}-01-01",
          "endDate": f"{end + 5}-12-31"
      }
  )

  r.raise_for_status()

  return pd.read_csv(StringIO(r.text))

# Consolidate data for all case studies into singular DataFrame

data = map(
    lambda study: get_data(**study),
    studies.iterrows()[1]
)

all_data = pd.concat(data)

# Empty header label is included in api response
all_data.drop(columns="Unnamed: 0", inplace=True)

USW00053917
USW00014819
USW00023293
USW00093820
USW00014739
USW00094728
USW00014762
USW00014922


# Export

The resulting DataFrame will be written to the "local" disk (Google Colab disk), where it can then be downloaded onto the client machine. The downloaded CSV can be uploaded as a Google Earth Engine table asset (with no geometry) for later use.

In [ ]:
all_data.to_csv("data.csv", index = False)

files.download("data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>